# Klasifikasi DemogPairs Menggunakan ViT (Wajah dan Umur) & SVM

In [1]:
import numpy as np
import utils as u
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from tqdm import tqdm

joblib.parallel_backend('threading')

## Load Dataset

In [2]:
data = u.load_demogpairs()
pd.DataFrame(data)

,db_code,image_path,full_path,label,label_idx
0,CWF,able_wanamakok/002.jpg,dataset/demogpairs/images\able_wanamakok/002.jpg,Asian_Females,5
1,CWF,able_wanamakok/004.jpg,dataset/demogpairs/images\able_wanamakok/004.jpg,Asian_Females,5
2,CWF,able_wanamakok/007.jpg,dataset/demogpairs/images\able_wanamakok/007.jpg,Asian_Females,5
3,CWF,able_wanamakok/008.jpg,dataset/demogpairs/images\able_wanamakok/008.jpg,Asian_Females,5
4,CWF,able_wanamakok/012.jpg,dataset/demogpairs/images\able_wanamakok/012.jpg,Asian_Females,5
...,...,...,...,...,...
10795,CWF,zachary_quinto/177.jpg,dataset/demogpairs/images\zachary_quinto/177.jpg,White_Males,3
10796,CWF,zachary_quinto/214.jpg,dataset/demogpairs/images\zachary_quinto/214.jpg,White_Males,3
10797,CWF,zachary_quinto/217.jpg,dataset/demogpairs/images\zachary_quinto/217.jpg,White_Males,3
10798,CWF,zachary_quinto/218.jpg,dataset/demogpairs/images\zachary_quinto/218.jpg,White_Males,3


## Load Fitur

In [3]:
face_features = joblib.load('features/demogpairs_vit-age.pkl')
emotion_features = joblib.load('features/demogpairs_vit-face.pkl')
features = {}
for d in tqdm(data):
    key = d['image_path']
    features[key] = np.array(list(emotion_features[key]) + list(face_features[key]))
print('Jumlah fitur per gambar:', np.array(features[list(features.keys())[0]]).shape[0])

  0%|          | 0/10800 [00:00<?, ?it/s]

  5%|▌         | 588/10800 [00:00<00:01, 5873.96it/s]

 13%|█▎        | 1387/10800 [00:00<00:01, 7116.46it/s]

 20%|█▉        | 2106/10800 [00:00<00:01, 7147.70it/s]

 27%|██▋       | 2961/10800 [00:00<00:01, 7700.71it/s]

...

 81%|████████▏ | 8777/10800 [00:01<00:00, 8176.36it/s]

 89%|████████▉ | 9652/10800 [00:01<00:00, 8347.68it/s]

 97%|█████████▋| 10487/10800 [00:01<00:00, 8110.24it/s]

100%|██████████| 10800/10800 [00:01<00:00, 7988.03it/s]

Jumlah fitur per gambar: 1536


## Split Data

In [4]:
X = np.array([features[d['image_path']] for d in data])
y = np.array([d['label_idx'] for d in data])
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
print((len(X_train), len(X_test)))

(8640, 2160)


## Kombinasi Parameter

In [5]:
from sklearn.decomposition import PCA

grid_params = [
    {
        'scaler': [None, MinMaxScaler()],  # EN: scaling option / ID: opsi scaling
        'pca': [None, PCA(n_components=0.5), PCA(n_components=0.75)],  # EN: with and without PCA / ID: dengan dan tanpa PCA
        
        'classifier': [SVC()],  # EN: SVM model / ID: model SVM
        
        'classifier__C': [0.01, 0.1, 1, 10],  # EN: regularization / ID: regularisasi
        'classifier__kernel': ['rbf', 'poly', 'linear'],  # EN: kernel types / ID: jenis kernel
        
        'classifier__gamma': ['scale', 'auto'],  # EN: gamma / ID: gamma
        'classifier__degree': [2, 3],  # EN: degree / ID: derajat
        
        'classifier__tol': [1e-3],  # EN: tolerance / ID: toleransi
        'classifier__probability': [True],  # EN: probability / ID: probabilitas
    },
]

pipeline = Pipeline(steps=[
    ('scaler', None),  # EN: optional scaler / ID: scaler opsional
    ('pca', None),     # EN: optional PCA / ID: PCA opsional
    ('classifier', None)  # EN: classifier / ID: classifier
])

skv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy', 
    'f1': 'f1_macro', 
    'precision': 'precision_macro', 
    'recall': 'recall_macro'
}

grid_models = {}
for params in grid_params:
    key = str(params['classifier'][0]).split('(')[0]
    grid_models[key] = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        cv=skv, refit='accuracy',
        scoring=scoring, n_jobs=int(joblib.cpu_count() * 0.6),
        verbose=1, error_score='raise',
        return_train_score=True
    )
    print(f'{key}: {len(ParameterGrid(params))} kombinasi')

SVC: 288 kombinasi


## Klasifikasi

In [6]:
evaluation_results, fold_results = u.evaluate_models(
    grid_models,
    X_train, y_train,
    X_test, y_test,
    target_names=u.demogpairs_classes,
    model_prefix='models/clf_demogpairs_svm_vit-face-age_',
    results_path='results/demogpairs_svm_vit-face-age_'
)

sorted_results = pd.DataFrame(evaluation_results).sort_values(by='test_accuracy', ascending=False).to_dict('records')
u.html_br()
_dtable = u.display_table(sorted_results)

Evaluating: SVC


{'classifier': 'SVC', 'classifier__C': 10, 'classifier__degree': 2, 'classifier__gamma': 'scale', 'classifier__kernel': 'poly', 'classifier__probability': True, 'classifier__tol': 0.001, 'pca': None, 'scaler': None}


Accuracy  : 0.9254629629629629
Precision : 0.9254124492919567
Recall    : 0.925462962962963
F1 Score  : 0.9254074085625507
               precision    recall  f1-score   support

Asian_Females     0.9091    0.9167    0.9129       360
  Asian_Males     0.9258    0.9361    0.9309       360
Black_Females     0.9148    0.8944    0.9045       360
  Black_Males     0.9389    0.9389    0.9389       360
White_Females     0.9190    0.9139    0.9164       360
  White_Males     0.9449    0.9528    0.9488       360

     accuracy                         0.9255      2160
    macro avg     0.9254    0.9255    0.9254      2160
 weighted avg     0.9254    0.9255    0.9254      2160



Class,OvR Accuracy,Precision,Recall,F1-Score,Support
Asian_Females,0.9708333333333333,0.9090909090909091,0.9166666666666666,0.9128630705394191,360
Asian_Males,0.9768518518518519,0.9258241758241759,0.9361111111111111,0.930939226519337,360
Black_Females,0.9685185185185186,0.9147727272727273,0.8944444444444445,0.9044943820224719,360
Black_Males,0.9796296296296296,0.9388888888888889,0.9388888888888889,0.9388888888888889,360
White_Females,0.9722222222222222,0.9189944134078212,0.9138888888888889,0.9164345403899721,360
White_Males,0.9828703703703704,0.9449035812672176,0.9527777777777777,0.9488243430152142,360


Confusion matrix saved: images\cm_svm_vit-face-age_SVC.png



Confusion Matrix:
                         Asian_Females       Asian_Males     Black_Females       Black_Males     White_Females       White_Males
       Asian_Females               330                 7                 9                 0                12                 2
         Asian_Males                 9               337                 3                 6                 0                 5
       Black_Females                 9                 3               322                10                14                 2
         Black_Males                 0                 7                 9               338                 0                 6
       White_Females                15                 2                 9                 0               329                 5
         White_Males                 0                 8                 0                 6                 3               343


model_name,model_file_path,best_parameters,test_accuracy,test_f1,test_precision,test_recall,parameter_combinations
SVC,models/clf_demogpairs_svm_vit-face-age_SVC.pkl,"{'classifier': 'SVC', 'classifier__C': 10, 'classifier__degree': 2, 'classifier__gamma': 'scale', 'classifier__kernel': 'poly', 'classifier__probability': True, 'classifier__tol': 0.001, 'pca': None, 'scaler': None}",0.9254629629629629,0.9254074085625507,0.9254124492919567,0.925462962962963,288


In [7]:
model, training_time = u.load_object('models/clf_demogpairs_svm_vit-face-age_SVC.pkl')
u.h(5, 'Waktu Pelatihan (Jobs)')
u.seconds_to_time(round(training_time))

{'input_seconds': 22197.0,
 'days': 0,
 'hours': 6,
 'minutes': 9,
 'seconds': 57.0,
 'text': '0 hari 6 jam 9 menit 57.0 detik'}

In [8]:
u.h(5, 'Waktu Pelatihan')
times = [fr['Train Time Mean'] * 5 for fr in fold_results]
u.seconds_to_time(round(np.sum(times) + model.refit_time_))

{'input_seconds': 180684.0,
 'days': 2,
 'hours': 2,
 'minutes': 11,
 'seconds': 24.0,
 'text': '2 hari 2 jam 11 menit 24.0 detik'}

In [9]:
def remove_keys(data, keys_to_remove, inplace=False):
    # EN: if not inplace, create a copy / ID: jika tidak inplace, buat salinan
    if not inplace:
        data = data.copy()
    
    # EN: loop through keys and remove safely / ID: loop key dan hapus dengan aman
    for key in keys_to_remove:
        data.pop(key, None)  # EN: avoid error if key not found / ID: aman jika key tidak ada
    
    return data

def dict_to_sentence(d):
    # EN: convert key-value pairs into readable parts / ID: ubah key-value jadi bagian kalimat
    parts = [f"{k}={v}" for k, v in d.items()]
    
    # EN: join all parts into one sentence / ID: gabungkan jadi satu kalimat
    sentence = ", ".join(parts)
    
    return sentence

fold_displays = []
for r in [remove_keys(r, ['No', 'F1 Score Mean', 'Precision Mean', 'Recall Mean', 'Train Time Mean']) for r in fold_results]:
    r['Params'] = dict_to_sentence(remove_keys(r['Params'], ['classifier'])).replace('classifier__', '')
    r['Mean'] = r['Accuracy Mean']
    del r['Accuracy Mean']
    fold_displays.append(r)
fold_displays = [{'No': idx + 1, **r} for idx, r in enumerate(sorted(fold_displays, key=lambda x: x['Mean'], reverse=True))]
_dtable = u.display_table(fold_displays, n_items=[3, 3, 3, 3], column_widths=['5%', '65%', '5%', '5%', '5%', '5%', '5%', '5%'])

No,Params,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Mean
1,"C=10, degree=2, gamma=scale, kernel=poly, probability=True, tol=0.001, pca=None, scaler=None",0.93,0.9207,0.9149,0.923,0.9265,0.923
2,"C=10, degree=3, gamma=scale, kernel=rbf, probability=True, tol=0.001, pca=None, scaler=None",0.9282,0.9207,0.9144,0.9207,0.9271,0.9222
3,"C=10, degree=2, gamma=scale, kernel=rbf, probability=True, tol=0.001, pca=None, scaler=None",0.9282,0.9207,0.9144,0.9207,0.9271,0.9222
...,...,...,...,...,...,...,...
96,"C=1, degree=3, gamma=scale, kernel=linear, probability=True, tol=0.001, pca=PCA, scaler=MinMaxScaler",0.9005,0.8924,0.8895,0.901,0.8895,0.8946
97,"C=1, degree=2, gamma=auto, kernel=poly, probability=True, tol=0.001, pca=PCA, scaler=None",0.9039,0.8941,0.8854,0.897,0.8918,0.8944
98,"C=1, degree=3, gamma=auto, kernel=linear, probability=True, tol=0.001, pca=PCA, scaler=None",0.8987,0.8941,0.8895,0.8883,0.8935,0.8928
...,...,...,...,...,...,...,...
191,"C=0.01, degree=2, gamma=auto, kernel=linear, probability=True, tol=0.001, pca=PCA, scaler=None",0.875,0.8675,0.8623,0.8657,0.8738,0.8689
192,"C=0.01, degree=3, gamma=auto, kernel=linear, probability=True, tol=0.001, pca=PCA, scaler=None",0.875,0.8675,0.8623,0.8657,0.8738,0.8689
